In [2]:
import numpy as np
import tensorflow as tf
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax import jax_utils
import optax
from flax.training import train_state
import time

from Preprocess import prepare_tcn_data
from TCN_model import TCN, init_train_state, train_step_tpu

I0000 00:00:1776663067.228982   68385 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776663067.230211   68385 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776663067.279478   68385 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776663068.624892   68385 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [3]:
with tf.io.gfile.GFile("gs://fi2010-lob-data/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Training/Train_Dst_NoAuction_ZScore_CF_1.txt", 'r') as f:
    raw_data = np.loadtxt(f)

In [4]:
batch_size = 1024
time_step = 127

In [5]:
train_dataset = prepare_tcn_data(raw_data, batch_size, time_step, True, False)
train_iter = train_dataset.as_numpy_iterator()

E0000 00:00:1776663076.578111   68385 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
len(train_dataset)

38

## Test: Non repeated data pipeline + jax.pmap

### batch_size = 1024

In [9]:
# initial training settings
rng = jax.random.PRNGKey(3721)
rng, dropout_rng = jax.random.split(rng)
model = TCN()
state, dropout_rng = init_train_state(rng, model)
dropout_rngs = jax.random.split(dropout_rng, 4)
replicated_state = jax_utils.replicate(state)

# Data warmup
next_batch = next(train_iter)
parallel_x = next_batch[0].reshape(4, -1, 127, 40).astype(jnp.bfloat16)
parallel_x = jax.device_put(parallel_x)

parallel_y = next_batch[1].reshape(4, -1).astype(jnp.int32)
parallel_y = jax.device_put(parallel_y)

In [10]:
EPOCHS = 3

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    steps = 0
    
    train_iter = train_dataset.as_numpy_iterator()
    next_batch = next(train_iter)
    parallel_x = next_batch[0].reshape(4, -1, 127, 40).astype(jnp.bfloat16)
    parallel_x = jax.device_put(parallel_x)

    parallel_y = next_batch[1].reshape(4, -1).astype(jnp.int32)
    parallel_y = jax.device_put(parallel_y)
    
    start_time = time.time()
    for step in range(len(train_dataset)):

        replicated_state, loss, acc = train_step_tpu(replicated_state, parallel_x, parallel_y, dropout_rngs)
                
        epoch_loss += loss[0].item()
        steps += 1

        try:
            train_iter = train_dataset.as_numpy_iterator()
            next_batch = next(train_iter)
            parallel_x = next_batch[0].reshape(4, -1, 127, 40).astype(jnp.bfloat16)
            parallel_x = jax.device_put(parallel_x)

            parallel_y = next_batch[1].reshape(4, -1).astype(jnp.int32)
            parallel_y = jax.device_put(parallel_y)
        except StopIteration:
            break
        
        # 测试模式小贴士：每跑几步打印一下，确保模型没卡死，Loss 没变成 NaN
        if steps % 10 == 0:
            print(f"  ⚡ Step {steps} | 实时 Loss: {loss[0].item():.4f} | Acc: {acc[0].item():.4f}")

    avg_loss = epoch_loss / max(1, steps)
    print(f'time:{time.time() - start_time}')
    print(f"✅ Epoch {epoch+1}/{EPOCHS} 完美收官！")
    print(f"📊 总计运行步数: {steps} | 平均 Loss: {avg_loss:.4f}")
    print("-" * 50)

  ⚡ Step 10 | 实时 Loss: 1.8739 | Acc: 0.4512
  ⚡ Step 20 | 实时 Loss: 1.2183 | Acc: 0.4385
  ⚡ Step 30 | 实时 Loss: 1.0906 | Acc: 0.4648
time:46.65565896034241
✅ Epoch 1/3 完美收官！
📊 总计运行步数: 38 | 平均 Loss: 1.6960
--------------------------------------------------
  ⚡ Step 10 | 实时 Loss: 0.9815 | Acc: 0.5078
  ⚡ Step 20 | 实时 Loss: 0.9388 | Acc: 0.5420
  ⚡ Step 30 | 实时 Loss: 0.9281 | Acc: 0.5596
time:9.949470043182373
✅ Epoch 2/3 完美收官！
📊 总计运行步数: 38 | 平均 Loss: 0.9570
--------------------------------------------------
  ⚡ Step 10 | 实时 Loss: 0.9182 | Acc: 0.5625
  ⚡ Step 20 | 实时 Loss: 0.8626 | Acc: 0.6006
  ⚡ Step 30 | 实时 Loss: 0.8713 | Acc: 0.5918
time:9.919679880142212
✅ Epoch 3/3 完美收官！
📊 总计运行步数: 38 | 平均 Loss: 0.8866
--------------------------------------------------


### batch size = 4096

In [11]:
batch_size = 4096
time_step = 127

In [12]:
train_dataset = prepare_tcn_data(raw_data, batch_size, time_step, True, False)
train_iter = train_dataset.as_numpy_iterator()

In [13]:
len(train_dataset)

9

In [14]:
# initial training settings
rng = jax.random.PRNGKey(3721)
rng, dropout_rng = jax.random.split(rng)
model = TCN()
state, dropout_rng = init_train_state(rng, model)
dropout_rngs = jax.random.split(dropout_rng, 4)
replicated_state = jax_utils.replicate(state)

# Data warmup
next_batch = next(train_iter)
parallel_x = next_batch[0].reshape(4, -1, 127, 40).astype(jnp.bfloat16)
parallel_x = jax.device_put(parallel_x)

parallel_y = next_batch[1].reshape(4, -1).astype(jnp.int32)
parallel_y = jax.device_put(parallel_y)

In [15]:
EPOCHS = 3

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    steps = 0
    
    train_iter = train_dataset.as_numpy_iterator()
    next_batch = next(train_iter)
    parallel_x = next_batch[0].reshape(4, -1, 127, 40).astype(jnp.bfloat16)
    parallel_x = jax.device_put(parallel_x)

    parallel_y = next_batch[1].reshape(4, -1).astype(jnp.int32)
    parallel_y = jax.device_put(parallel_y)
    
    start_time = time.time()
    for step in range(len(train_dataset)):

        replicated_state, loss, acc = train_step_tpu(replicated_state, parallel_x, parallel_y, dropout_rngs)
                
        epoch_loss += loss[0].item()
        steps += 1

        try:
            train_iter = train_dataset.as_numpy_iterator()
            next_batch = next(train_iter)
            parallel_x = next_batch[0].reshape(4, -1, 127, 40).astype(jnp.bfloat16)
            parallel_x = jax.device_put(parallel_x)

            parallel_y = next_batch[1].reshape(4, -1).astype(jnp.int32)
            parallel_y = jax.device_put(parallel_y)
        except StopIteration:
            break
        
        # 测试模式小贴士：每跑几步打印一下，确保模型没卡死，Loss 没变成 NaN
        if steps % 10 == 0:
            print(f"  ⚡ Step {steps} | 实时 Loss: {loss[0].item():.4f} | Acc: {acc[0].item():.4f}")

    avg_loss = epoch_loss / max(1, steps)
    print(f'time:{time.time() - start_time}')
    print(f"✅ Epoch {epoch+1}/{EPOCHS} 完美收官！")
    print(f"📊 总计运行步数: {steps} | 平均 Loss: {avg_loss:.4f}")
    print("-" * 50)

time:32.534499406814575
✅ Epoch 1/3 完美收官！
📊 总计运行步数: 9 | 平均 Loss: 3.2867
--------------------------------------------------
time:3.24764084815979
✅ Epoch 2/3 完美收官！
📊 总计运行步数: 9 | 平均 Loss: 1.4128
--------------------------------------------------
time:3.28353214263916
✅ Epoch 3/3 完美收官！
📊 总计运行步数: 9 | 平均 Loss: 1.1537
--------------------------------------------------
